# This is an experimental notebook

In [1]:
# necessary imports
import pandas as pd 
from pathlib import Path
import wfdb
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, Flatten, Dense, GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    cohen_kappa_score
    
)
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold

c:\Users\ZEYNEP\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
import sys
from pathlib import Path

# Add src directory to path
src_path = Path("../").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to path: {src_path}")

Added to path: C:\Users\ZEYNEP\OneDrive\Desktop\CTG_Dissertation


In [3]:
# paths 
raw_dataset = Path("data/raw_dataset")

records = [p.stem for p in raw_dataset.glob("*.hea")]
print(f"Found {len(records)} CTG records")


Found 552 CTG records


In [4]:
# Read one record to test
record = wfdb.rdrecord(raw_dataset / records[0])

signals = record.p_signal        
signal_names = record.sig_name   
fs = record.fs                  
print(f"Signal names: {signal_names}, Sampling rate: {fs} Hz")

Signal names: ['FHR', 'UC'], Sampling rate: 4 Hz


In [5]:
# Convert signals to a list
fhr = signals[:, 0].tolist()
uc = signals[:, 1].tolist()
print(f"First 10 FHR values: {fhr[:10]}")
print(f"First 10 UC values: {uc[:10]}")

First 10 FHR values: [150.5, 150.5, 151.0, 151.25, 151.25, 150.25, 150.25, 150.25, 148.75, 148.75]
First 10 UC values: [7.0, 8.5, 8.5, 7.5, 9.5, 8.5, 10.5, 12.0, 11.0, 11.5]


In [6]:
'''
Step 2: Signal cleaning 
Repeated zero signals at the end of the samples were removed
'''
def remove_trailing_zeros(signal):
    """Remove trailing zeros from a signal."""
    if not isinstance(signal, list):
        raise ValueError("Input signal must be a list.")
    
    # Find the index of the last non-zero element
    last_non_zero_index = len(signal) - 1
    while last_non_zero_index >= 0 and signal[last_non_zero_index] == 0:
        last_non_zero_index -= 1
    
    # Return the signal up to the last non-zero element
    return signal[:last_non_zero_index + 1]

In [7]:
'''
Step 3: Signal Extraction
The 30 minutes immediately preceding the last non-zero signals were extracted.
'''
def extract_last_30_minutes(signal, sampling_rate=4):
    ''' reject short recordings and extract last 30 minutes of signal '''
    num_samples_30_minutes = 30 * 60 * sampling_rate

    if len(signal) < num_samples_30_minutes:
        return []  # reject short recordings

    return signal[-num_samples_30_minutes:]


In [8]:
'''
Step 4: Downsample signals to 1 Hz
The signals were originally sampled at 4 Hz. They were downsampled to 1 Hz by taking every fourth sample.
Paper explicitly states: “Signals were downsampled to 1 Hz for 30 minutes (1800 points)”
'''
def downsample_to_1hz(signal, original_fs=4, target_fs=1):
    factor = original_fs // target_fs
    return signal[::factor]


In [9]:
'''
Step 5: Selection
Only cases that satisfied the selection criteria (specifically, a signal loss less than 16%) were used for the final analysis
'''
def is_signal_acceptable(signal, threshold=0.16):
    """Check if the signal loss is within the acceptable threshold."""
    
    # Reject empty or invalid signals
    if not signal or len(signal) == 0:
        return False
    
    total_length = len(signal)
    zero_count = signal.count(0)
    signal_loss = zero_count / total_length
    
    return signal_loss < threshold


In [10]:
''' 
Step 6: Perform the above steps on the dataset 
'''
processed_records = []

for rec in records:
    record = wfdb.rdrecord(raw_dataset / rec)
    signals = record.p_signal

    fhr = signals[:, 0].tolist()
    uc  = signals[:, 1].tolist()

    # Cleaning 
    fhr = remove_trailing_zeros(fhr)
    uc  = remove_trailing_zeros(uc)

    # Extraction 
    fhr = extract_last_30_minutes(fhr, sampling_rate=record.fs)
    uc  = extract_last_30_minutes(uc, sampling_rate=record.fs)

    # Downsampling
    fhr = downsample_to_1hz(fhr, original_fs=fs, target_fs=1)
    uc  = downsample_to_1hz(uc, original_fs=fs, target_fs=1)

    # Selection
    if is_signal_acceptable(fhr) and is_signal_acceptable(uc):
        processed_records.append({
            "rec_id": rec,
            "FHR": fhr,
            "UC": uc
        })
print(f"Processed {len(processed_records)} records after cleaning and selection.")

Processed 220 records after cleaning and selection.


In [11]:
''' 
Classification: Map signals to step3_labels based on record IDs
'''
# path to majority voting labels
Labels = 'ExpertAnnotations\step3_labels.csv'

# print number of normal and abnormal deliveries
labels_df = pd.read_csv(Labels)
num_normal = sum(labels_df['Clinical_Label'] == 'No Hypoxia (Normal)')
num_suspicious = sum(labels_df['Clinical_Label'] == 'Mild Hypoxia (Suspicious)')
num_severe = sum(labels_df['Clinical_Label'] == 'Severe Hypoxia (Pathological)')
num_uninterpretable = sum(labels_df['Clinical_Label'] == 'Uninterpretable (Filtered)')
print(f'Number of normal deliveries: {num_normal}')
print(f'Number of suspicious deliveries: {num_suspicious}')
print(f'Number of severe deliveries: {num_severe}')
print(f'Number of uninterpretable deliveries: {num_uninterpretable}')

Number of normal deliveries: 127
Number of suspicious deliveries: 153
Number of severe deliveries: 57
Number of uninterpretable deliveries: 215


In [12]:
data_df = pd.DataFrame(processed_records)
print(data_df.shape)

(220, 3)


In [13]:
'''
Map the labels
'''

# Create a mapping from record ID to clinical label
labels_df = pd.read_csv(Labels)
print(labels_df.columns)

data_df['rec_id'] = data_df['rec_id'].astype(str)
labels_df['rec_id'] = labels_df['rec_id'].astype(str)
merged_df = data_df.merge(
    labels_df,
    left_on="rec_id",
    right_on="rec_id",
    how="inner"
)

print(merged_df.shape)
label_counts = merged_df['Clinical_Label'].value_counts()
print(label_counts)


Index(['rec_id', 'Majority_Vote_Label', 'Clinical_Label'], dtype='object')
(220, 5)
Clinical_Label
Uninterpretable (Filtered)       91
Mild Hypoxia (Suspicious)        57
No Hypoxia (Normal)              42
Severe Hypoxia (Pathological)    30
Name: count, dtype: int64


In [14]:
UNINT = "Uninterpretable (Filtered)"
MAP_3 = {
    "No Hypoxia (Normal)": 0,
    "Mild Hypoxia (Suspicious)": 1,
    "Severe Hypoxia (Pathological)": 2
}

# 1) Build X
X = np.array([
    np.stack([row["FHR"], row["UC"]], axis=0)
    for _, row in merged_df.iterrows()
], dtype=np.float32)[..., np.newaxis]   # (N,2,1800,1)

# 2) Build y_interp (1=interpretable, 0=uninterpretable)
y_interp = (merged_df["Clinical_Label"] != UNINT).astype(int).values.astype(np.int32)

# 3) Build y_sev (0/1/2 for interpretable, -1 for uninterpretable)
def map_sev(lbl):
    if lbl == UNINT:
        return -1
    return MAP_3[lbl]

y_sev = merged_df["Clinical_Label"].apply(map_sev).values.astype(np.int32)

print("X:", X.shape)
print("Severity counts:", dict(zip(*np.unique(y_sev, return_counts=True))))
print("Interp counts:", dict(zip(*np.unique(y_interp, return_counts=True))))


X: (220, 2, 1800, 1)
Severity counts: {-1: 91, 0: 42, 1: 57, 2: 30}
Interp counts: {0: 91, 1: 129}


In [21]:
'''
Convert severity to ordinal targets + weights
'''
# ordinal targets (N,2)
y_clip = np.clip(y_sev, 0, 2)
y_ord = np.stack([(y_clip > 0), (y_clip > 1)], axis=1).astype(np.float32)

# weights: ignore severity loss for uninterpretable
w_sev = (y_sev != -1).astype(np.float32)
w_interp = np.ones_like(w_sev, dtype=np.float32)

# interp target needs shape (N,1)
y_interp_f = y_interp.astype(np.float32).reshape(-1,1)


# Build Model

In [22]:
# Define better ordinal decoding function
def decode_ordinal_better(y_ord_pred):
    """
    Decode ordinal predictions using probability-based approach.
    y_ord_pred: (N, 2) with sigmoid outputs
    Returns class 0, 1, or 2 based on ordinal constraints
    """
    p_ge1 = y_ord_pred[:, 0]  # P(class >= 1)
    p_ge2 = y_ord_pred[:, 1]  # P(class >= 2)
    
    # Enforce constraint: p_ge1 >= p_ge2
    p_ge2 = np.minimum(p_ge2, p_ge1)
    
    # Compute class probabilities
    p_0 = 1 - p_ge1
    p_1 = p_ge1 - p_ge2
    p_2 = p_ge2
    
    # Stack probabilities and pick class with highest probability
    probs = np.stack([p_0, p_1, p_2], axis=1)
    return np.argmax(probs, axis=1)

In [23]:
from keras.layers import Concatenate

def build_ctg_ms_encoder(input_shape=(2, 1800, 1), dropout_rate=0.25):
    inputs = Input(shape=input_shape)

    x = Conv2D(4, (1,3), padding="same", use_bias=False)(inputs)
    x = BatchNormalization()(x)

    x = DepthwiseConv2D((2,1), depth_multiplier=2, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D((1,4))(x)
    x = Dropout(dropout_rate)(x)

    b1 = SeparableConv2D(8, (1,3), padding="same", use_bias=False)(x)
    b1 = BatchNormalization()(b1); b1 = Activation("relu")(b1)

    b2 = SeparableConv2D(8, (1,7), padding="same", use_bias=False)(x)
    b2 = BatchNormalization()(b2); b2 = Activation("relu")(b2)

    b3 = SeparableConv2D(8, (1,15), padding="same", use_bias=False)(x)
    b3 = BatchNormalization()(b3); b3 = Activation("relu")(b3)

    x = Concatenate(axis=-1)([b1,b2,b3])

    x = Conv2D(8, (1,1), use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = AveragePooling2D((1,4))(x)
    x = Dropout(dropout_rate)(x)

    feat = GlobalAveragePooling2D()(x)
    return inputs, feat


def build_multitask_model():
    inputs, feat = build_ctg_ms_encoder()
    severity_ord = Dense(2, activation="sigmoid", name="severity_ord")(feat)
    interpretable = Dense(1, activation="sigmoid", name="interpretable")(feat)
    return Model(inputs, [severity_ord, interpretable])


In [24]:
def ordinal_bce_loss(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred), axis=-1)

In [25]:
'''
Prepare to store evaluation metrics and to perform 10-fold cross-validation
'''
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Separate normal and abnormal indices
abnormal_indices = np.where(y_interp == 0)[0]
normal_indices = np.where(y_interp == 1)[0]

n_iterations = 10
test_ratio = 0.1  # 9:1 split

auc_scores = []
f1_scores = []
precision_scores = []
recall_scores = []
sensitivity_scores = []
accuracy_scores = []
thresholds_list = [0.4,0.3,0.2,0.5]
all_y_true = []
all_y_prob = []

In [26]:
'''
Perform CV with Ordinal Multitask Learning
Interpretability head: learns on ALL samples (interpretable vs uninterpretable).
Severity head: learns only on interpretable samples (y_sev != -1) via sample weights.
'''

# Mask for interpretable samples (for severity task)
mask_interp = (y_sev != -1)

# Outer loop: independent experiments
for iteration in range(1, n_iterations + 1):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration}/{n_iterations}")
    print(f"{'='*60}")
    
    # Use ALL samples for interpretability head
    all_indices = np.arange(len(X))
    X_iter = X[all_indices]
    y_ord_iter = y_ord[all_indices]
    y_interp_iter = y_interp_f[all_indices]   # shape (N,1)
    y_sev_iter = y_sev[all_indices]
    
    # Stratify folds on interpretability label (0 = uninterpretable, 1 = interpretable)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + iteration)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_iter, y_interp_iter.reshape(-1)), 1):
        print(f"\n--- Fold {fold}/5 ---")
        
        X_train = X_iter[train_idx]
        y_ord_train = y_ord_iter[train_idx]
        y_interp_train = y_interp_iter[train_idx]
        y_sev_train = y_sev_iter[train_idx]
        
        X_test = X_iter[test_idx]
        y_ord_test = y_ord_iter[test_idx]
        y_interp_test = y_interp_iter[test_idx]
        y_sev_test = y_sev_iter[test_idx]
        
        # ---------- Sample weights ----------
        # 1) Severity: only interpretable samples contribute (y_sev != -1)
        w_sev_train = (y_sev_train != -1).astype(np.float32)
        
        # Optional: per-class weighting for severity on interpretable subset
        y_sev_train_interp = y_sev_train[y_sev_train != -1]
        if len(y_sev_train_interp) > 0:
            class_counts = np.bincount(y_sev_train_interp, minlength=3)
            class_weights_sev = {i: len(y_sev_train_interp) / (3 * max(count, 1))
                                 for i, count in enumerate(class_counts)}
            w_sev_class = np.ones_like(w_sev_train, dtype=np.float32)
            for c, w_c in class_weights_sev.items():
                w_sev_class[y_sev_train == c] = w_c
            w_sev_train = w_sev_train * w_sev_class
        
        # 2) Interpretability: class-balanced weights for 0/1
        y_interp_train_flat = y_interp_train.reshape(-1).astype(int)
        interp_counts = np.bincount(y_interp_train_flat, minlength=2)
        total_interp = len(y_interp_train_flat)
        w_interp_0 = total_interp / (2 * max(interp_counts[0], 1))
        w_interp_1 = total_interp / (2 * max(interp_counts[1], 1))
        w_interp_train = np.where(
            y_interp_train_flat == 1,
            w_interp_1,
            w_interp_0
        ).astype(np.float32).reshape(-1, 1)
        
        # ---------- Split info (by severity label) ----------
        n_normal_train = (y_sev_train == 0).sum()
        n_mild_train = (y_sev_train == 1).sum()
        n_severe_train = (y_sev_train == 2).sum()
        n_unint_train = (y_sev_train == -1).sum()
        
        n_normal_test = (y_sev_test == 0).sum()
        n_mild_test = (y_sev_test == 1).sum()
        n_severe_test = (y_sev_test == 2).sum()
        n_unint_test = (y_sev_test == -1).sum()
        
        print(f"Train: Normal={n_normal_train}, Mild={n_mild_train}, Severe={n_severe_train}, Unint={n_unint_train}")
        print(f"Test:  Normal={n_normal_test}, Mild={n_mild_test}, Severe={n_severe_test}, Unint={n_unint_test}")
        
        # ---------- Build and train model ----------
        K.clear_session()
        model = build_multitask_model()
        
        model.compile(
            optimizer=Adam(1e-3),
            loss=[ordinal_bce_loss, "binary_crossentropy"],
            loss_weights=[0.5, 1.0]  # more weight on interpretability head
        )
        
        early_stop = EarlyStopping(
            monitor="loss",
            patience=25,
            restore_best_weights=True,
            verbose=0
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor="loss",
            factor=0.5,
            patience=15,
            min_lr=1e-7,
            verbose=0
        )
        
        print("Training...", end=" ", flush=True)
        history = model.fit(
            X_train,
            [y_ord_train, y_interp_train],
            sample_weight=[w_sev_train, w_interp_train],
            batch_size=8,
            epochs=300,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        print(f"Done! ({len(history.history['loss'])} epochs)")
        
        # ---------- Evaluate interpretability head (ALL test samples) ----------
        sev_pred_ord, interp_pred = model.predict(X_test, verbose=0)
        interp_prob = interp_pred.reshape(-1)
        interp_hat = (interp_prob >= 0.5).astype(int)
        
        try:
            interp_acc = accuracy_score(y_interp_test.reshape(-1), interp_hat)
            interp_f1 = f1_score(y_interp_test.reshape(-1), interp_hat, zero_division=0)
            
            if len(np.unique(y_interp_test)) == 2:
                interp_auc = roc_auc_score(y_interp_test.reshape(-1), interp_prob)
            else:
                interp_auc = np.nan
            
            print(f"Interpretability - Acc: {interp_acc:.3f}, F1: {interp_f1:.3f}, AUC: {interp_auc:.3f}")
        except Exception as e:
            print(f"Interpretability eval error: {e}")
        
        # ---------- Evaluate severity head (interpretable only) ----------
        mask_test_interp = (y_sev_test != -1)
        if mask_test_interp.sum() > 0:
            y_sev_test_interp = y_sev_test[mask_test_interp]
            sev_pred_ord_interp = sev_pred_ord[mask_test_interp]
            
            def decode_ordinal(y_ord_pred, thr=0.5):
                return (y_ord_pred >= thr).sum(axis=1).astype(int)
            
            try:
                print("  Severity (threshold-based):")
                for thr in [0.3, 0.4, 0.5]:
                    y_pred_sev = decode_ordinal(sev_pred_ord_interp, thr=thr)
                    sev_acc = accuracy_score(y_sev_test_interp, y_pred_sev)
                    sev_f1 = f1_score(y_sev_test_interp, y_pred_sev, average="macro", zero_division=0)
                    sev_qwk = cohen_kappa_score(y_sev_test_interp, y_pred_sev, weights="quadratic")
                    print(f"    thr={thr}: Acc={sev_acc:.3f}, F1={sev_f1:.3f}, QWK={sev_qwk:.3f}")
                
                print("  Severity (probability-based):")
                y_pred_sev_prob = decode_ordinal_better(sev_pred_ord_interp)
                sev_acc_prob = accuracy_score(y_sev_test_interp, y_pred_sev_prob)
                sev_f1_prob = f1_score(y_sev_test_interp, y_pred_sev_prob, average="macro", zero_division=0)
                sev_qwk_prob = cohen_kappa_score(y_sev_test_interp, y_pred_sev_prob, weights="quadratic")
                print(f"    Acc={sev_acc_prob:.3f}, F1={sev_f1_prob:.3f}, QWK={sev_qwk_prob:.3f}")
            except Exception as e:
                print(f"Severity eval error: {e}")

print("\n" + "="*60)
print("Cross-validation completed!")
print("="*60)


Iteration 1/10

--- Fold 1/5 ---
Train: Normal=30, Mild=46, Severe=27, Unint=73
Test:  Normal=12, Mild=11, Severe=3, Unint=18
Training... Done! (160 epochs)
Interpretability - Acc: 0.545, F1: 0.412, AUC: 0.607
  Severity (threshold-based):
    thr=0.3: Acc=0.423, F1=0.381, QWK=0.348
    thr=0.4: Acc=0.538, F1=0.381, QWK=0.325
    thr=0.5: Acc=0.615, F1=0.422, QWK=0.232
  Severity (probability-based):
    Acc=0.423, F1=0.198, QWK=-0.058

--- Fold 2/5 ---
Train: Normal=37, Mild=43, Severe=23, Unint=73
Test:  Normal=5, Mild=14, Severe=7, Unint=18
Training... Done! (186 epochs)
Interpretability - Acc: 0.477, F1: 0.378, AUC: 0.712
  Severity (threshold-based):
    thr=0.3: Acc=0.423, F1=0.354, QWK=0.215
    thr=0.4: Acc=0.346, F1=0.266, QWK=0.207
    thr=0.5: Acc=0.308, F1=0.233, QWK=0.145
  Severity (probability-based):
    Acc=0.231, F1=0.157, QWK=0.043

--- Fold 3/5 ---
Train: Normal=32, Mild=46, Severe=25, Unint=73
Test:  Normal=10, Mild=11, Severe=5, Unint=18
Training... Done! (83 epo

In [27]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.metrics import cohen_kappa_score
import numpy as np

# ---------- 1) Predict ----------
sev_pred_ord, interp_pred = model.predict(X, verbose=0)   # sev_pred_ord: (N,2), interp_pred: (N,1)
interp_prob = interp_pred.reshape(-1)
interp_hat = (interp_prob >= 0.5).astype(int)

# ---------- 2) Interpretability metrics (ALL samples) ----------
print("=== Interpretability (ALL) ===")
print("Acc:", accuracy_score(y_interp, interp_hat))
print("F1 :", f1_score(y_interp, interp_hat, zero_division=0))

# AUC requires both classes present
if len(np.unique(y_interp)) == 2:
    print("AUC:", roc_auc_score(y_interp, interp_prob))
else:
    print("AUC: N/A (only one class present)")

print("Confusion:\n", confusion_matrix(y_interp, interp_hat))
print("Report:\n", classification_report(y_interp, interp_hat, zero_division=0))


# ---------- 3) Ordinal decode helper ----------
def decode_ordinal(y_ord_pred, thr=0.5):
    """
    For K=3 classes => 2 ordinal probs.
    class = number of thresholds passed.
    """
    return (y_ord_pred >= thr).sum(axis=1).astype(int)


# ---------- 4) Severity metrics (interpretable only) ----------
mask = (y_sev != -1)  # interpretable samples only
ytrue = y_sev[mask]

print("\nInterpretable severity N:", mask.sum())
print("True class counts:", dict(zip(*np.unique(ytrue, return_counts=True))))

for thr in [0.3, 0.4, 0.5]:
    yhat = decode_ordinal(sev_pred_ord, thr=thr)[mask]

    print(f"\n=== Severity (interpretable only) thr={thr} ===")
    print("Pred class counts:", dict(zip(*np.unique(yhat, return_counts=True))))

    acc = accuracy_score(ytrue, yhat)
    macro_f1 = f1_score(ytrue, yhat, average="macro", zero_division=0)
    qwk = cohen_kappa_score(ytrue, yhat, weights="quadratic")

    print("Acc     :", acc)
    print("Macro-F1:", macro_f1)
    print("QWK     :", qwk)
    print("Confusion:\n", confusion_matrix(ytrue, yhat))
    print("Report:\n", classification_report(ytrue, yhat, zero_division=0))


=== Interpretability (ALL) ===
Acc: 0.6090909090909091
F1 : 0.6228070175438597
AUC: 0.6940114149416474
Confusion:
 [[63 28]
 [58 71]]
Report:
               precision    recall  f1-score   support

           0       0.52      0.69      0.59        91
           1       0.72      0.55      0.62       129

    accuracy                           0.61       220
   macro avg       0.62      0.62      0.61       220
weighted avg       0.64      0.61      0.61       220


Interpretable severity N: 129
True class counts: {0: 42, 1: 57, 2: 30}

=== Severity (interpretable only) thr=0.3 ===
Pred class counts: {0: 5, 1: 116, 2: 8}
Acc     : 0.4496124031007752
Macro-F1: 0.298929589377453
QWK     : 0.10002718129926624
Confusion:
 [[ 2 38  2]
 [ 3 52  2]
 [ 0 26  4]]
Report:
               precision    recall  f1-score   support

           0       0.40      0.05      0.09        42
           1       0.45      0.91      0.60        57
           2       0.50      0.13      0.21        30

    accu

In [ ]:
from sklearn.metrics import roc_auc_score

# Convert ordinal outputs to class probabilities (rough, but useful)
# P(class >=1) = sev_pred_ord[:,0]
# P(class >=2) = sev_pred_ord[:,1]
p_ge1 = sev_pred_ord[:, 0]
p_ge2 = sev_pred_ord[:, 1]

if mask.sum() > 0:
    # Severe vs rest
    y_severe = (y_sev == 2).astype(int)
    if len(np.unique(y_severe[mask])) == 2:
        print("\nAUC Severe vs Rest:", roc_auc_score(y_severe[mask], p_ge2[mask]))
    else:
        print("\nAUC Severe vs Rest: N/A")

    # (Mild+Severe) vs Normal
    y_abnormal = (y_sev >= 1).astype(int)
    if len(np.unique(y_abnormal[mask])) == 2:
        print("AUC Abnormal vs Normal:", roc_auc_score(y_abnormal[mask], p_ge1[mask]))
    else:
        print("AUC Abnormal vs Normal: N/A")



AUC Severe vs Rest: 0.6737373737373736
AUC Abnormal vs Normal: 0.6453201970443351


In [23]:
# At the end of CV loop, before "Cross-validation completed!"

# Aggregate metrics across all folds
print("\n" + "="*60)
print("OVERALL CV RESULTS")
print("="*60)

# Interpretability stats
print("\nInterpretability Task:")
print(f"  Mean AUC: {np.mean(interp_auc_list):.3f} ± {np.std(interp_auc_list):.3f}")
print(f"  Mean F1:  {np.mean(interp_f1_list):.3f} ± {np.std(interp_f1_list):.3f}")

# Severity stats per threshold
for thr in [0.3, 0.4, 0.5]:
    qwk_vals = [scores[thr]['qwk'] for scores in severity_scores]
    acc_vals = [scores[thr]['acc'] for scores in severity_scores]
    
    print(f"\nSeverity (thr={thr}):")
    print(f"  Mean QWK: {np.mean(qwk_vals):.3f} ± {np.std(qwk_vals):.3f}")
    print(f"  Mean Acc: {np.mean(acc_vals):.3f} ± {np.std(acc_vals):.3f}")


OVERALL CV RESULTS

Interpretability Task:


NameError: name 'interp_auc_list' is not defined